In [316]:
import numpy as np 
import time
import pickle
import random
import math
import mmap
import os
from tqdm import tqdm 

In [317]:
#Parameters

n_head = 1 #the number of attention heads
n_layer = 2 # the number of decoders
input_dim = 100 # Aka n_embed
block_size = 16
batch_size = 8
max_sequence_length = 100
lr = 1.4e-2
epochs = 100
evals = 1

In [318]:
class Tokenizer:#fine but not used cause prefered caracter level tokenizer
    def __init__(self, vocab_size=10000):
        self.vocab_size = vocab_size
        self.word_to_idx = {}
        self.idx_to_word = {}
        self.pad_token = '<pad>'
        self.unk_token = '<unk>'
        self.add_special_tokens()

    def add_special_tokens(self):
        self.word_to_idx[self.pad_token] = 0
        self.word_to_idx[self.unk_token] = 1
        self.idx_to_word[0] = self.pad_token
        self.idx_to_word[1] = self.unk_token

    def __call__(self, text):
        return self.fit_on_texts(text)

    def fit_on_texts(self, texts):
        if isinstance(texts, str):
            texts = [texts]  # Convert single string to a list of strings
        # Extract all unique words from the texts
        all_words = set()
        
        for text in texts:
            all_words.update(text.split())

        # Sort the words by frequency and select the top vocab_size - 2 words
        word_counts = {word: 0 for word in all_words}
        for text in texts:
            for word in text.split():
                word_counts[word] += 1
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        top_words = [word for word, _ in sorted_words[:self.vocab_size - 2]]

        # Assign indices to the words
        self.word_to_idx.update({word: i + 2 for i, word in enumerate(top_words)})
        self.idx_to_word.update({i + 2: word for i, word in enumerate(top_words)})



    def encode_with_lengths(self, text):
        words = text.split()
        encoded = [self.word_to_idx.get(word, 1) for word in words]
        sentence_lengths = [len(sentence.split()) for sentence in text.split('.')]
        return encoded, sentence_lengths

    def decode(self, encoded, sentence_lengths):
        words = []
        start = 0
        for i, length in enumerate(sentence_lengths):
            sentence_tokens = encoded[start:start+length]
            sentence_words = [self.idx_to_word.get(idx, self.unk_token) for idx in sentence_tokens]
            words.extend(sentence_words)
            if i < len(sentence_lengths) - 1:
                words.append('.')
            start += length
        return ' '.join(words)

In [319]:
class PositionalEncoding:
    def __init__(self, input_dim, max_sequence_length=max_sequence_length):
        self.input_dim = input_dim
        self.max_sequence_length = max_sequence_length
        self.PE = None
        
    def __call__(self, index):
        return self.forward(index)

    def forward(self, index):
        even_i = np.arange(0, self.input_dim, 2)
        denominator = np.array([math.pow(10000, i/self.input_dim) for i in even_i])
        position = index.reshape(-1, 1)  # Convert index to positions
        even_PE = np.sin(position/denominator)
        odd_PE = np.cos(position/denominator)
        stacked = np.stack([even_PE, odd_PE], axis=2)
        PE = np.reshape(stacked, (len(index), self.input_dim))
        return PE

In [320]:
def parse(params, name):
    for param_name, param_value in params.items():
        if param_name == name:
            return param_value
        elif isinstance(param_value, dict):
            return parse(param_value, name)
        
# # 1. Fix AdamOptim function, brokennnnnn😑😑😑😑😑😑😑😑😑😑😑😑😑😑😑😑😑😑😑
# def AdamOptim(model, lr=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-8):
#     # Initialize or get existing optimizer state
#     if not hasattr(model, 'optimizer_state'):
#         model.optimizer_state = {
#             'm_params': {},
#             'v_params': {},
#             't': {}
#         }
    
#     params = model.get_params()
#     grads = model.get_grad()
    
#     for param_name, param_value in params.items():
#         if param_value is None:
#             continue
            
#         # Get corresponding gradient
#         grad_name = f'grad_{param_name}'
#         if grad_name not in grads:
#             continue
            
#         grad = grads[grad_name]
#         if grad is None:
#             continue
        
#         # Ensure gradient shape matches parameter shape
#         if grad.shape != param_value.shape:
#             print(f"Shape mismatch: {param_name} param {param_value.shape} vs grad {grad.shape}")
#             continue
            
#         # Initialize optimizer state for this parameter
#         if param_name not in model.optimizer_state['m_params']:
#             model.optimizer_state['m_params'][param_name] = np.zeros_like(param_value)
#             model.optimizer_state['v_params'][param_name] = np.zeros_like(param_value)
#             model.optimizer_state['t'][param_name] = 0
        
#         # Update step counter
#         model.optimizer_state['t'][param_name] += 1
        
#         # Update moment estimates
#         model.optimizer_state['m_params'][param_name] = (
#             beta_1 * model.optimizer_state['m_params'][param_name] + (1 - beta_1) * grad
#         )
#         model.optimizer_state['v_params'][param_name] = (
#             beta_2 * model.optimizer_state['v_params'][param_name] + (1 - beta_2) * (grad ** 2)
#         )
        
#         # Bias correction
#         m_hat = model.optimizer_state['m_params'][param_name] / (1 - beta_1 ** model.optimizer_state['t'][param_name])
#         v_hat = model.optimizer_state['v_params'][param_name] / (1 - beta_2 ** model.optimizer_state['t'][param_name])
        
#         # Update parameter in-place
#         param_value -= lr * m_hat / (np.sqrt(v_hat) + epsilon)


In [321]:
class Block:
    def __init__(self, input_dim, sequence_length, n_head, lr=lr):
        if input_dim % n_head != 0:
            raise ValueError("Input_dim must be divisible by n_head")
        self.input_dim = input_dim
        self.n_head = n_head
        self.head_size = input_dim // n_head
        self.sa = MultiHeadAttention(input_dim, sequence_length, head_size=self.head_size, num_heads=n_head)
        self.ffwd = FeedForward(input_dim)
        self.ln1 = LayerNormalization(input_dim)  # Pass input_dim as normalized_shape
        self.ln2 = LayerNormalization(input_dim)  # Pass input_dim as normalized_shape

        
    def __call__(self, x, apply_mask):
        return self.forward(x, apply_mask)
    
    def get_params(self):
        return {'sa': self.sa.get_params(), 
                'ffwd': self.ffwd.get_params(), 
                'ln1': self.ln1.get_params(),
                'ln2': self.ln2.get_params(),
               }
        
    def get_grad(self):
        return {
                'grad_sa': self.sa.get_grad(), 
                'grad_ffwd': self.ffwd.get_grad(), 
                'grad_ln1': self.ln1.get_grad(), 
                'grad_ln2': self.ln2.get_grad()
               }

    def zero_grad(self):
        # Reset all gradients to zero
        self.sa.zero_grad()
        self.ffwd.zero_grad()
        self.ln1.zero_grad()
        self.ln2.zero_grad()
        
    def parameters(self):
        return self.get_params()
    
    def forward(self, x, apply_mask):
        y = self.sa(x, apply_mask)
        
        y = self.ln1(x+y) #apply residual connection

        z = self.ffwd(y)
        
        out = self.ln2(z+y) #apply residual connection
        self.out = out
        return out
        
    def backward(self, dL_dy): #dL_dy represens the gradient output taken as parameter 
        
        # Backward pass through the second Layer Normalization
        dL_dx = self.ln2.backward(dL_dy)
        # print("After backprop through ln2: ", dL_dx.shape, len(dL_dln2))
        
        # Backward pass through the Feed Forward network
        dL_dy = dL_dx
        dL_dx = self.ffwd.backward(dL_dy, self.ffwd.output_activation)
        # print("After backprop through ffwd: ", dL_dx.shape, len(dL_dffwd))

        # Backward pass through the first Layer Normalization
        dL_dy = dL_dx
        dL_dx = self.ln1.backward(dL_dy)
        
        # Backward pass through the Self Attention mechanism
        dL_dy = dL_dx
        dL_dx = self.sa.backward(dL_dy)
        
       
        return dL_dx

    def update(self):
        self.sa.update()
        self.ffwd.update()
        self.ln1.update()
        self.ln2.update()

In [322]:

class MultiHeadAttention:
    # Multiple Heads of self-Attention in parallel 
    
    def __init__(self, input_dim, sequence_length, head_size, num_heads, lr=lr):
        self.heads = ([Head(input_dim = input_dim, sequence_length=sequence_length, head_size=head_size, lr=lr) for _ in range(num_heads)])# type: ignore
        self.n_heads = num_heads
        self.proj = Linear(head_size * num_heads, input_dim, lr=lr)# type: ignore
        
        # Actually head_size*num_heads is equal to n_embd but by proceeding like this we add another learnable param the bias
    
    def __call__(self, x, apply_mask):
        return self.forward(x, apply_mask)
        
    def get_grad(self):
        grads = {'grad_proj': self.proj.get_grad()}
        for i, head in enumerate(self.heads):
            grads[f'grad_head_{i}'] = head.get_grad()
        return grads
        
    def zero_grad(self):
        # Reset all gradients to zero
        for head in self.heads:
            head.zero_grad()
        self.proj.zero_grad()
        
    def get_params(self):
        params = {
                  'proj': self.proj.get_params()
                 }
        for i, head in enumerate(self.heads):
            params[f'head_{i}'] = head.get_params()
        
        return params
        
    def parameters(self):
        return self.get_params()
    
    def forward(self, x, apply_mask):
        out = np.concatenate([h(x, apply_mask) for h in self.heads], axis=-1) # concatenate along the (batch_size, sqlength, F): F been the feature dimension 
        out = self.proj(out)  # Pass the concatenated output through the projection layer
        self.out = out
        return out

    def backward(self, grad_out):
        grad_proj = self.proj.backward(grad_out)

        # Backpropagate through individual attention heads
        grad_proj_split = np.split(grad_proj, self.n_heads, axis=-1)
        heads = [head.backward(gp) for head, gp in zip(self.heads, grad_proj_split)]
        for i, h in enumerate(heads):
            out = np.concatenate([h for h in heads], axis=-1)
        return out
        
    def update(self):
        for head in self.heads:
            head.update()
        self.proj.update()
        



In [323]:
class Head:
    def __init__(self, head_size, input_dim, sequence_length, mask=None, lr=0.001, bias=True):
        self.input_dim = input_dim
        self.seq_length = sequence_length
        self.head_size = head_size
        self.Q = Linear(input_dim, head_size, bias, lr=lr)
        self.K = Linear(input_dim, head_size, bias, lr=lr)
        self.V = Linear(input_dim, head_size, bias, lr=lr)
        self.linear_layer = Linear(head_size, input_dim, bias, lr=lr)
        self.lr = lr
        self.bias = bias
        
        self.mask = self.set_mask(mask)
        
        # Cache for backward pass
        self.x = None
        self.q = None
        self.k = None
        self.v = None
        self.scaled_scores = None
        self.attention = None
        self.values = None
        self.out = None

    def softmax(self, x):
        """Numerically stable softmax"""
        # Apply along the last dimension (over keys)
        x_max = np.max(x, axis=-1, keepdims=True)
        x_shifted = x - x_max
        exp_x = np.exp(x_shifted)
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)
    
    def __call__(self, x, apply_mask=True):
        return self.forward(x, apply_mask)
    
    def get_params(self):
        params = {
            'Q': self.Q.get_params(), 
            'K': self.K.get_params(),
            'V': self.V.get_params(),
            'linear_layer': self.linear_layer.get_params()
        }
        return params
    
    def get_grad(self, name=None):
        grads = {
            'grad_Q': self.Q.get_grad(), 
            'grad_K': self.K.get_grad(), 
            'grad_V': self.V.get_grad(), 
            'grad_linear_layer': self.linear_layer.get_grad()
        }
        return grads[f'grad_{name}'] if name is not None else grads
        
    def parameters(self):
        return self.get_params()
        
    def zero_grad(self):
        """Reset all gradients to zero"""
        self.Q.zero_grad()
        self.K.zero_grad()
        self.V.zero_grad()
        self.linear_layer.zero_grad()
        
    def set_mask(self, mask=None):
        """Create or set causal mask"""
        if mask is not None:
            self.mask = mask
            return mask
            
        # Create lower triangular mask for causal attention
        mask = np.tril(np.ones((self.seq_length, self.seq_length)))
        # Convert to attention mask: 0 -> -inf, 1 -> 0
        mask = np.where(mask == 0, -np.inf, 0.0)
        self.mask = mask
        return mask
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Compute scaled dot-product attention
        Q, K, V: (batch_size, seq_len, head_size)
        Returns: output (batch_size, seq_len, head_size), attention_weights (batch_size, seq_len, seq_len)
        """
        batch_size, seq_len, head_size = Q.shape
        dk = head_size
        
        # Compute attention scores: Q @ K^T / sqrt(dk)
        # Use einsum for clarity: 'bih,bjh->bij' means batch matrix multiply
        scores = np.einsum('bih,bjh->bij', Q, K) / math.sqrt(dk)
        
        # Apply mask if provided
        if mask is not None:
            scores = scores + mask  # mask should have -inf where attention is not allowed
        
        # Store for backward pass
        self.scaled_scores = scores
        
        # Apply softmax to get attention weights
        attention_weights = self.softmax(scores)
        
        # Apply attention to values: attention @ V
        output = np.einsum('bij,bjh->bih', attention_weights, V)
        
        return output, attention_weights
    
    def softmax_backward(self, grad_output, softmax_output):
        """
        Backward pass through softmax
        grad_output: gradient w.r.t. softmax output (B, T, T)
        softmax_output: the softmax output (B, T, T)
        Returns: gradient w.r.t. softmax input (B, T, T)
        """
        batch_size, seq_len, _ = grad_output.shape
        grad_input = np.zeros_like(grad_output)
        
        for b in range(batch_size):
            for i in range(seq_len):
                # For each row of the softmax
                s_i = softmax_output[b, i, :]  # (T,)
                grad_i = grad_output[b, i, :]   # (T,)
                
                # Jacobian of softmax: J_ij = s_i * (δ_ij - s_j)
                # grad_input[i] = sum_j(J_ij * grad_output[j])
                #               = s_i * (grad_output[i] - sum_j(s_j * grad_output[j]))
                grad_input[b, i, :] = s_i * (grad_i - np.sum(s_i * grad_i))
        
        return grad_input
    
    def scaled_dot_product_attention_backward(self, grad_output):
        """
        Backward pass through scaled dot-product attention
        grad_output: gradient w.r.t attention output (B, T, head_size)
        Returns: grad_Q, grad_K, grad_V
        """
        batch_size, seq_len, head_size = grad_output.shape
        dk = head_size
        
        # Step 1: Gradient w.r.t V
        # output = attention @ V, so ∂L/∂V = attention^T @ grad_output
        grad_V = np.einsum('bij,bih->bjh', self.attention, grad_output)
        
        # Step 2: Gradient w.r.t attention weights
        # output = attention @ V, so ∂L/∂attention = grad_output @ V^T
        grad_attention = np.einsum('bih,bjh->bij', grad_output, self.v)
        
        # Step 3: Gradient through softmax
        grad_scores = self.softmax_backward(grad_attention, self.attention)
        
        # Step 4: Scale by 1/sqrt(dk)
        grad_scores = grad_scores / math.sqrt(dk)
        
        # Step 5: Gradient w.r.t Q and K
        # scores = Q @ K^T, so ∂L/∂Q = grad_scores @ K, ∂L/∂K = grad_scores^T @ Q
        grad_Q = np.einsum('bij,bjh->bih', grad_scores, self.k)
        grad_K = np.einsum('bji,bih->bjh', grad_scores, self.q)
        
        return grad_Q, grad_K, grad_V
    
    def forward(self, x, apply_mask=True): 
        """Forward pass through attention head"""
        self.x = x  # Store input for backward pass
        
        # Compute Q, K, V projections
        self.q = self.Q(x)  # (B, T, head_size)
        self.k = self.K(x)  # (B, T, head_size) 
        self.v = self.V(x)  # (B, T, head_size)

        # Apply scaled dot-product attention
        mask = self.mask if apply_mask else None
        self.values, self.attention = self.scaled_dot_product_attention(
            self.q, self.k, self.v, mask
        )
                
        # Final linear projection
        self.out = self.linear_layer(self.values)
        
        return self.out

    def backward(self, grad_output):
        """Backward pass through attention head"""
        if grad_output is None:
            return None
            
        # Ensure grad_output has the right shape
        if grad_output.shape != self.out.shape:
            grad_output = grad_output.reshape(self.out.shape)
        
        # Step 1: Backward through final linear layer
        grad_values = self.linear_layer.backward(grad_output)
        
        if grad_values is None:
            raise ValueError("grad_values is None after linear layer backward pass")
        
        # Step 2: Backward through attention mechanism
        grad_Q, grad_K, grad_V = self.scaled_dot_product_attention_backward(grad_values)
        
        # Step 3: Backward through Q, K, V projections
        grad_x_Q = self.Q.backward(grad_Q)
        grad_x_K = self.K.backward(grad_K) 
        grad_x_V = self.V.backward(grad_V)
        
        # Step 4: Sum gradients w.r.t input x
        grad_x = grad_x_Q + grad_x_K + grad_x_V
        
        return grad_x

    def update(self):
        """Update all parameters"""
        self.Q.update()
        self.K.update()
        self.V.update()
        self.linear_layer.update()


In [324]:

class FeedForward:
    def __init__(self, input_dim, lr=0.001, bias=True, tol=1e-6):
        assert isinstance(input_dim, int), "Input size must be an integer"
        assert input_dim > 0, "Input size must be positive"
        assert isinstance(lr, (int, float)), "Learning rate must be a number"
        assert lr > 0, "Learning rate must be positive"

        # Create layers and activation
        self.layers = [
            Linear(input_dim, 4 * input_dim, bias, lr=lr),
            ReLU(),  # Add explicit ReLU activation
            Linear(4 * input_dim, input_dim, bias, lr=lr),
        ]
        
        self.input_dim = input_dim
        self.hidden_size = 4 * input_dim
        self.output_size = input_dim
        self.lr = lr
        self.bias = bias
        self.tol = tol
        
        # Cache for backward pass
        self.activations = []
        
    def __call__(self, x):
        return self.forward(x)
        
    def parameters(self):
        return self.get_params()
    
    def get_params(self):
        params = {}
        learnable_layers = []
        for layer in self.layers:
            if hasattr(layer, 'get_params'):  # Only get params from learnable layers
                learnable_layers.append(layer.get_params())
            
        params['layers'] = learnable_layers
        return params
        
    def get_grad(self, name=None):
        grad_layers = []
        for layer in self.layers:
            if hasattr(layer, 'get_grad'):  # Only get grads from learnable layers
                grad_layers.append(layer.get_grad())
        
        self.grads = {'grad_layers': grad_layers}
        return self.grads[f'grad_{name}'] if name is not None else self.grads

    def zero_grad(self):
        for layer in self.layers:
            if hasattr(layer, 'zero_grad'):  # Only zero grad for learnable layers
                layer.zero_grad()
        
    def forward(self, x):
        self.activations = []
        current_input = x
        
        # Forward pass through all layers
        for layer in self.layers:
            current_input = layer(current_input)
            self.activations.append(current_input)
        
        return current_input
    
    def backward(self, grad_output):
        if grad_output is None:
            return None
        
        current_grad = grad_output
        
        # Backward pass through layers in reverse order
        for layer in reversed(self.layers):
            current_grad = layer.backward(current_grad)
        
        return current_grad
        
    def update(self):
        for layer in self.layers:
            if hasattr(layer, 'update'):  # Only update learnable layers
                layer.update()


# # Alternative implementation with explicit activation handling
# class FeedForwardExplicit:
#     def __init__(self, input_dim, lr=0.001, bias=True, tol=1e-6):
#         assert isinstance(input_dim, int), "Input size must be an integer"
#         assert input_dim > 0, "Input size must be positive"
#         assert isinstance(lr, (int, float)), "Learning rate must be a number"
#         assert lr > 0, "Learning rate must be positive"

#         # Only Linear layers - handle activation explicitly
#         self.linear1 = Linear(input_dim, 4 * input_dim, bias, lr=lr)
#         self.linear2 = Linear(4 * input_dim, input_dim, bias, lr=lr)
        
#         self.input_dim = input_dim
#         self.hidden_size = 4 * input_dim
#         self.output_size = input_dim
#         self.lr = lr
#         self.bias = bias
#         self.tol = tol
        
#         # Cache for backward pass
#         self.x = None
#         self.z1 = None  # Linear output before activation
#         self.a1 = None  # After ReLU activation
#         self.z2 = None  # Final linear output
        
#     def __call__(self, x):
#         return self.forward(x)
        
#     def parameters(self):
#         return self.get_params()
    
#     def get_params(self):
#         return {
#             'linear1': self.linear1.get_params(),
#             'linear2': self.linear2.get_params()
#         }
        
#     def get_grad(self, name=None):
#         grads = {
#             'grad_linear1': self.linear1.get_grad(),
#             'grad_linear2': self.linear2.get_grad()
#         }
#         return grads[f'grad_{name}'] if name is not None else grads

#     def zero_grad(self):
#         self.linear1.zero_grad()
#         self.linear2.zero_grad()

#     def relu(self, x):
#         return np.maximum(x, 0)
    
#     def relu_derivative(self, x):
#         return (x > 0).astype(float)
        
#     def forward(self, x):
#         self.x = x
        
#         # First linear layer
#         self.z1 = self.linear1(x)
        
#         # ReLU activation
#         self.a1 = self.relu(self.z1)
        
#         # Second linear layer
#         self.z2 = self.linear2(self.a1)
        
#         return self.z2
    
#     def backward(self, grad_output):
#         if grad_output is None:
#             return None
        
#         # Backward through second linear layer
#         grad_a1 = self.linear2.backward(grad_output)
        
#         # Backward through ReLU activation
#         grad_z1 = grad_a1 * self.relu_derivative(self.z1)
        
#         # Backward through first linear layer
#         grad_input = self.linear1.backward(grad_z1)
        
#         return grad_input
        
#     def update(self):
#         self.linear1.update()
#         self.linear2.update()


# # Even more explicit version showing the mathematical operations
# class FeedForwardMath:
#     def __init__(self, input_dim, lr=0.001, bias=True, tol=1e-6):
#         assert isinstance(input_dim, int), "Input size must be an integer"
#         assert input_dim > 0, "Input size must be positive"
#         assert isinstance(lr, (int, float)), "Learning rate must be a number"
#         assert lr > 0, "Learning rate must be positive"

#         self.input_dim = input_dim
#         self.hidden_size = 4 * input_dim
#         self.output_size = input_dim
#         self.lr = lr
#         self.bias = bias
#         self.tol = tol
        
#         # Initialize weights with Glorot uniform
#         limit1 = np.sqrt(6 / (input_dim + 4 * input_dim))
#         limit2 = np.sqrt(6 / (4 * input_dim + input_dim))
        
#         self.W1 = np.random.uniform(-limit1, limit1, (input_dim, 4 * input_dim))
#         self.W2 = np.random.uniform(-limit2, limit2, (4 * input_dim, input_dim))
        
#         if bias:
#             self.b1 = np.zeros(4 * input_dim)
#             self.b2 = np.zeros(input_dim)
#         else:
#             self.b1 = None
#             self.b2 = None
            
#         # Gradients
#         self.grad_W1 = None
#         self.grad_W2 = None
#         self.grad_b1 = None
#         self.grad_b2 = None
        
#         # Cache for backward pass
#         self.x = None
#         self.z1 = None
#         self.a1 = None
#         self.z2 = None
        
#     def __call__(self, x):
#         return self.forward(x)
        
#     def parameters(self):
#         params = {'W1': self.W1, 'W2': self.W2}
#         if self.bias:
#             params.update({'b1': self.b1, 'b2': self.b2})
#         return params
        
#     def get_grad(self, name=None):
#         grads = {'grad_W1': self.grad_W1, 'grad_W2': self.grad_W2}
#         if self.bias:
#             grads.update({'grad_b1': self.grad_b1, 'grad_b2': self.grad_b2})
#         return grads[f'grad_{name}'] if name is not None else grads

#     def zero_grad(self):
#         self.grad_W1 = np.zeros_like(self.W1)
#         self.grad_W2 = np.zeros_like(self.W2)
#         if self.bias:
#             self.grad_b1 = np.zeros_like(self.b1)
#             self.grad_b2 = np.zeros_like(self.b2)

#     def relu(self, x):
#         return np.maximum(x, 0)
    
#     def relu_derivative(self, x):
#         return (x > 0).astype(float)
        
#     def forward(self, x):
#         self.x = x
        
#         # Handle different input shapes
#         original_shape = x.shape
#         if x.ndim > 2:
#             batch_size = np.prod(original_shape[:-1])
#             x_reshaped = x.reshape(batch_size, self.input_dim)
#         else:
#             x_reshaped = x
        
#         # First linear layer: z1 = xW1 + b1
#         self.z1 = np.dot(x_reshaped, self.W1)
#         if self.bias:
#             self.z1 += self.b1
            
#         # ReLU activation: a1 = ReLU(z1)
#         self.a1 = self.relu(self.z1)
        
#         # Second linear layer: z2 = a1W2 + b2
#         self.z2 = np.dot(self.a1, self.W2)
#         if self.bias:
#             self.z2 += self.b2
        
#         # Reshape back to original shape if needed
#         if x.ndim > 2:
#             self.z2 = self.z2.reshape(original_shape[:-1] + (self.output_size,))
        
#         return self.z2
    
#     def backward(self, grad_output):
#         if grad_output is None:
#             return None
            
#         # Handle different gradient shapes
#         original_grad_shape = grad_output.shape
#         if grad_output.ndim > 2:
#             batch_size = np.prod(original_grad_shape[:-1])
#             grad_output_reshaped = grad_output.reshape(batch_size, self.output_size)
#         else:
#             grad_output_reshaped = grad_output
            
#         # Gradient w.r.t. second layer parameters
#         self.grad_W2 = np.dot(self.a1.T, grad_output_reshaped)
#         if self.bias:
#             self.grad_b2 = np.sum(grad_output_reshaped, axis=0)
            
#         # Gradient w.r.t. a1
#         grad_a1 = np.dot(grad_output_reshaped, self.W2.T)
        
#         # Gradient w.r.t. z1 (through ReLU)
#         grad_z1 = grad_a1 * self.relu_derivative(self.z1)
        
#         # Gradient w.r.t. first layer parameters
#         x_reshaped = self.x.reshape(-1, self.input_dim) if self.x.ndim > 2 else self.x
#         self.grad_W1 = np.dot(x_reshaped.T, grad_z1)
#         if self.bias:
#             self.grad_b1 = np.sum(grad_z1, axis=0)
            
#         # Gradient w.r.t. input
#         grad_input = np.dot(grad_z1, self.W1.T)
        
#         # Reshape back to original input shape if needed
#         if self.x.ndim > 2:
#             grad_input = grad_input.reshape(self.x.shape)
        
#         return grad_input
        
#     def update(self):
#         if self.grad_W1 is not None:
#             self.W1 -= self.lr * self.grad_W1
#         if self.grad_W2 is not None:
#             self.W2 -= self.lr * self.grad_W2
#         if self.bias:
#             if self.grad_b1 is not None:
#                 self.b1 -= self.lr * self.grad_b1
#             if self.grad_b2 is not None:
#                 self.b2 -= self.lr * self.grad_b2

In [325]:
class LayerNormalization:
    def __init__(self, normalized_shape, epsilon=1e-5, tol=1e-9, lr=0.001):
        self.lr = lr
        self.epsilon = epsilon
        self.tol = tol
        self.normalized_shape = normalized_shape
        self.gamma = np.ones(normalized_shape) 
        self.beta = np.zeros(normalized_shape)
        self.grad_gamma = None
        self.grad_beta = None
        
        # Cache variables for backward pass
        self.x = None
        self.mean = None
        self.var = None
        self.std = None
        self.y = None
    
    def __call__(self, x):
        return self.forward(x)
        
    def parameters(self):
        return self.get_params()
        
    def get_params(self):
        return {
            'gamma': self.gamma,
            'beta': self.beta, 
        }
    
    def zero_grad(self):
        # Reset all gradients to zero
        self.grad_beta = np.zeros_like(self.beta)
        self.grad_gamma = np.zeros_like(self.gamma)
        
    def get_grad(self, name=None):
        self.grads = {
            'grad_gamma': self.grad_gamma, 
            'grad_beta': self.grad_beta
        }
        
        return self.grads[f'grad_{name}'] if name is not None else self.grads

    def forward(self, x):
        self.x = x
        
        # Compute mean and variance over the last dimension(s)
        # For normalized_shape, we normalize over the last len(normalized_shape) dimensions
        if isinstance(self.normalized_shape, int):
            self.dims = -1
        else:
            # For multi-dimensional normalized_shape, normalize over last N dimensions
            self.dims = tuple(range(-len(self.normalized_shape), 0))
        
        self.mean = x.mean(axis=self.dims, keepdims=True)
        self.var = ((x - self.mean) ** 2).mean(axis=self.dims, keepdims=True)
        self.std = np.sqrt(self.var + self.epsilon)
        self.std = np.maximum(self.std, self.tol)

        self.y = (x - self.mean) / self.std
        out = self.gamma * self.y + self.beta
        return out
        
    def backward(self, grad_output):
        if grad_output is None:
            return None
            
        N = grad_output.shape[self.dims] if isinstance(self.dims, int) else np.prod([grad_output.shape[d] for d in self.dims])
        
        # Gradient w.r.t. gamma and beta
        if isinstance(self.dims, int):
            # Sum over all dimensions except the normalized dimension
            sum_dims = tuple(i for i in range(grad_output.ndim) if i != self.dims)
        else:
            # Sum over all dimensions except the normalized dimensions
            sum_dims = tuple(i for i in range(grad_output.ndim) if i not in self.dims)
        
        self.grad_gamma = np.sum(grad_output * self.y, axis=sum_dims, keepdims=False)
        self.grad_beta = np.sum(grad_output, axis=sum_dims, keepdims=False)
        
        # Ensure gradients match parameter shapes
        if self.grad_gamma.shape != self.gamma.shape:
            self.grad_gamma = self.grad_gamma.reshape(self.gamma.shape)
        if self.grad_beta.shape != self.beta.shape:
            self.grad_beta = self.grad_beta.reshape(self.beta.shape)
        
        # Gradient w.r.t. input
        grad_y = grad_output * self.gamma
        
        # Intermediate gradients
        grad_var = np.sum(grad_y * (self.x - self.mean), axis=self.dims, keepdims=True) * (-0.5) * (self.var + self.epsilon) ** (-1.5)
        grad_mean = np.sum(grad_y * (-1.0 / self.std), axis=self.dims, keepdims=True) + grad_var * np.sum(-2.0 * (self.x - self.mean), axis=self.dims, keepdims=True) / N
        
        # Final gradient w.r.t. input
        grad_x = grad_y / self.std + grad_var * 2.0 * (self.x - self.mean) / N + grad_mean / N
        
        return grad_x

    def update(self):
        if self.grad_gamma is not None:
            self.gamma -= self.lr * self.grad_gamma
        if self.grad_beta is not None:
            self.beta -= self.lr * self.grad_beta


In [326]:
class Linear:
    def __init__(self, in_features, out_features, bias=True, lr=0.001):
        self.in_features = in_features
        self.out_features = out_features
        self.bias = bias
        self.lr = lr
        
        # Initialize weights with Glorot uniform initialization
        limit = np.sqrt(6 / (in_features + out_features))
        self.w = np.random.uniform(-limit, limit, (in_features, out_features))
        
        if self.bias:
            self.b = np.zeros(out_features)
        else:
            self.b = None

        # Cache variables
        self.x = None
        self.grad_w = None
        self.grad_b = None

    def __call__(self, x):
        return self.forward(x)
        
    def parameters(self):
        return self.get_params()
    
    def get_params(self):
        params = {
            'w': self.w,
            'b': self.b if self.bias else None
        }
        return params
        
    def zero_grad(self):
        # Reset all gradients to zero
        self.grad_w = np.zeros_like(self.w)
        if self.bias:
            self.grad_b = np.zeros_like(self.b)
            
    def get_grad(self, name=None):
        self.grads = {'grad_w': self.grad_w}
        if self.bias:
            self.grads['grad_b'] = self.grad_b

        return self.grads[f'grad_{name}'] if name is not None else self.grads
    
    def forward(self, x):
        self.x = x
        out = np.matmul(x, self.w)
        if self.bias:
            out += self.b
        return out
    
    def backward(self, grad_output):
        if grad_output is None:
            return None
        
        # Handle different input shapes flexibly
        original_shape = grad_output.shape
        
        # Reshape to 2D for matrix operations if needed
        if grad_output.ndim > 2:
            batch_size = np.prod(original_shape[:-1])
            grad_output_2d = grad_output.reshape(batch_size, self.out_features)
            x_2d = self.x.reshape(batch_size, self.in_features)
        else:
            grad_output_2d = grad_output
            x_2d = self.x
        
        # Gradient w.r.t. weights
        self.grad_w = np.dot(x_2d.T, grad_output_2d)
        
        # Gradient w.r.t. bias
        if self.bias:
            self.grad_b = np.sum(grad_output_2d, axis=0)
        
        # Gradient w.r.t. input
        grad_x = np.dot(grad_output_2d, self.w.T)
        
        # Reshape back to original input shape if needed
        if grad_output.ndim > 2:
            grad_x = grad_x.reshape(self.x.shape)
        
        return grad_x
    
    def update(self):
        if self.grad_w is not None:
            self.w -= self.lr * self.grad_w
        if self.bias and self.grad_b is not None:
            self.b -= self.lr * self.grad_b


class ReLU:
    """Separate ReLU activation class"""
    def __init__(self):
        self.x = None
    
    def __call__(self, x):
        return self.forward(x)
    
    def forward(self, x):
        self.x = x
        return np.maximum(x, 0)
    
    def backward(self, grad_output):
        if grad_output is None:
            return None
        return grad_output * (self.x > 0)

In [327]:
class Embedding:
    def __init__(self, vocab_size, input_dim, lr=0.001):
        self.vocab_size = vocab_size
        self.input_dim = input_dim
        self.lr = lr
        
        # Initialize embeddings with Glorot uniform initialization
        limit = np.sqrt(6 / (vocab_size + input_dim))
        self.embeddings = np.random.uniform(-limit, limit, (vocab_size, input_dim))
        
        # Initialize gradient accumulator
        self.grad_embeddings = np.zeros((vocab_size, input_dim))
        
        # Cache for backward pass
        self.last_indices = None

    def __call__(self, indices):
        return self.forward(indices)
    
    def get_params(self):
        return {
            'embeddings': self.embeddings,
        }
        
    def parameters(self):
        return self.get_params()
        
    def zero_grad(self):
        self.grad_embeddings = np.zeros_like(self.embeddings)
        
    def get_grad(self, name=None):
        self.grads = {'grad_embeddings': self.grad_embeddings}
        return self.grads[f'grad_{name}'] if name is not None else self.grads
        
    def forward(self, indices):
        self.last_indices = indices
        return self.embeddings[indices]

    def backward(self, grad_output):
        if grad_output is None or self.last_indices is None:
            return None
        
        # Handle both single index and array of indices
        indices = self.last_indices
        
        # Accumulate gradients (crucial for embeddings!)
        if isinstance(indices, (int, np.integer)):
            self.grad_embeddings[indices] += grad_output
        else:
            # For array of indices, use np.add.at for proper accumulation
            np.add.at(self.grad_embeddings, indices, grad_output)
        
        # Embeddings don't propagate gradients to their "input" (indices)
        # since indices are discrete and not differentiable
        return None
        
    def update(self):
        if self.grad_embeddings is not None:
            self.embeddings -= self.lr * self.grad_embeddings


In [328]:
#Stand Alone Methods

In [329]:
def txt_files_in_dir(directory):
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".txt") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

folder_path = "data"
output_file_train = "data/train_split.txt"
output_file_val = "data/val_split.txt"
vocab_file = "data/vocab.txt"

print(os.system("pwd"))

files = txt_files_in_dir(folder_path)
total_files = len(files)

#split_index = int(total_files * 0.9)
# files_train = files[:split_index]
# files_val = files[split_index:]

files_train = files[:100]
files_val = files[1:10]


vocab = set()

with open(output_file_train, 'w', encoding="utf-8") as outfile:
    for count, filename in enumerate(tqdm(files_train, total=len(files_train))):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'rt', encoding="utf-8") as infile:
            text = infile.read()
            outfile.write(text)
            character = set(text)
            vocab.update(character)
with open(output_file_val, 'w', encoding="utf-8") as outfile:
    for filename in tqdm(files_val, total=len(files_val)):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'rt', encoding='utf-8') as infile:
            text = infile.read()
            outfile.write(text)
            characters = set(text)
            vocab.update(characters)
            
with open(vocab_file, 'w', encoding="utf-8") as vfile:
    for char in vocab:
        vfile.write(char + '\n')


/home/cron/GITHUB/transformer-gpt-numpy
0


100%|██████████| 2/2 [00:00<00:00, 3586.41it/s]


In [330]:
#ok then i am going for caracter level
chars = ""
with open('data/vocab.txt', 'r', encoding='utf-8') as f:
    text=f.read()
    chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {char:i for i,char in enumerate(chars)}
itos = {i:char for i,char in enumerate(chars)}
encode  = lambda s:[stoi[c] for c in s]
decode = lambda l:"".join([itos[i] for i in l])


#memory map for using snippets of text from a single file of any size
def get_random_chunk(split):
    filename = "data/train_split.txt" if split == 'train' else "data/val_split.txt"
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            #determine file size and a random position to start reading
            
            file_size = len(mm)
            start_pos = random.randint(0, (file_size) - block_size*batch_size)
            
            #Seek to the random position and read the block of text
            mm.seek(start_pos)
            block = mm.read(block_size*batch_size-1)
            # block = mm.read(n*block_size*batch_size-1),  where we determine the text amount of text read 
            
            #decode the block to a string, ignoring any invalid byte sequences
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r', '')
            
            #Train and test splits
            data = np.array(encode(decoded_block)) 
    return data


def get_batch(split):
    data = get_random_chunk(split)
    ix = np.random.randint(len(data) - block_size, size=(batch_size,))
    x =  np.stack([data[i:i+block_size] for i in ix]) 
    y =  np.stack([data[i+1:i+block_size+1] for i in ix])# Appartir du next char
    return x, y
x, y = get_batch('train')

In [331]:
# tok = Tokenizer()
# tok.fit_on_texts(text)

# l, ll = tok.encode_with_lengths("The climate profile is taken from closest available")
# tok.decode(l, ll)

In [332]:
import numpy as np
import pickle

class GPT:
    def __init__(self, vocab_size, input_dim, block_size, n_head, n_layer, max_sequence_length=None, lr=3e-4):
        # Store hyperparameters
        self.vocab_size = vocab_size
        self.input_dim = input_dim
        self.block_size = block_size
        self.n_head = n_head
        self.n_layer = n_layer
        self.lr = lr
        
        # Set max_sequence_length for positional encoding
        if max_sequence_length is None:
            max_sequence_length = block_size
        self.max_sequence_length = max_sequence_length
        
        # Initialize components
        self.embedding_table = Embedding(vocab_size, input_dim, lr=lr)
        self.position_embedding_table = PositionalEncoding(max_sequence_length, input_dim)
        self.decoder_blocks = [
            Block(input_dim, sequence_length=block_size, n_head=n_head, lr=lr) 
            for _ in range(n_layer)
        ]
        self.ln_f = LayerNormalization(input_dim, lr=lr)
        self.lm_head = Linear(input_dim, vocab_size, lr=lr)
        
        # Cache for backward pass
        self.index = None
        self.tok_embed = None
        self.pos_embed = None
        self.x_after_embed = None
        self.block_outputs = []
        self.x_after_ln = None
        self.output = None

    def cross_entropy(self, logits, targets):
        """
        Compute cross-entropy loss
        logits: (batch_size * seq_len, vocab_size)
        targets: (batch_size * seq_len,) - integer indices
        """
        batch_size = logits.shape[0]
        
        # Apply softmax with numerical stability
        max_logits = np.max(logits, axis=-1, keepdims=True)
        exp_logits = np.exp(logits - max_logits)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        # Clip probabilities to avoid log(0)
        probs = np.clip(probs, 1e-12, 1.0)
        
        # Compute cross-entropy loss
        log_probs = np.log(probs[np.arange(batch_size), targets])
        loss = -np.mean(log_probs)
        
        return loss
    
    def forward(self, index, targets=None, apply_mask=True):
        """Forward pass through the GPT model"""
        self.index = index
        batch_size, seq_len = index.shape
        
        # Clear previous cache
        self.block_outputs = []
        
        # Token embeddings
        self.tok_embed = self.embedding_table(index)  # (batch_size, seq_len, input_dim)
        
        # Positional embeddings
        pos_indices = np.arange(seq_len)
        self.pos_embed = self.position_embedding_table(pos_indices)  # (seq_len, input_dim)
        
        # Broadcast positional embeddings to match batch size
        if self.pos_embed.ndim == 2:  # (seq_len, input_dim)
            self.pos_embed = np.expand_dims(self.pos_embed, axis=0)  # (1, seq_len, input_dim)
        
        # Add embeddings
        self.x_after_embed = self.tok_embed + self.pos_embed
        
        # Through transformer blocks
        x = self.x_after_embed
        for i, block in enumerate(self.decoder_blocks):
            x = block(x, apply_mask)
            self.block_outputs.append(x)  # Store intermediate outputs
        
        # Final layer norm
        self.x_after_ln = self.ln_f(x)
        
        # Language modeling head
        self.output = self.lm_head(self.x_after_ln)  # (batch_size, seq_len, vocab_size)
        
        # Compute loss if targets provided
        loss = None
        if targets is not None:
            # Reshape for loss calculation
            logits_flat = self.output.reshape(-1, self.vocab_size)  # (batch_size * seq_len, vocab_size)
            targets_flat = targets.reshape(-1)  # (batch_size * seq_len,)
            
            # Filter out padding tokens if any (assuming -1 or similar)
            valid_mask = targets_flat >= 0
            if np.any(valid_mask):
                logits_valid = logits_flat[valid_mask]
                targets_valid = targets_flat[valid_mask]
                loss = self.cross_entropy(logits_valid, targets_valid)
            else:
                loss = self.cross_entropy(logits_flat, targets_flat)
        
        return self.output, loss

    def softmax(self, x, axis=-1, keepdims=False):
        """Numerically stable softmax"""
        x_max = np.max(x, axis=axis, keepdims=True)
        exp_x = np.exp(x - x_max)
        sum_exp = np.sum(exp_x, axis=axis, keepdims=keepdims if keepdims else True)
        
        if not keepdims and sum_exp.ndim > exp_x.ndim:
            sum_exp = np.squeeze(sum_exp, axis=axis)
        
        return exp_x / sum_exp

    def zero_grad(self):
        """Reset all gradients to zero"""
        self.embedding_table.zero_grad()
        # Note: PositionalEncoding typically doesn't have learnable parameters
        for block in self.decoder_blocks:
            block.zero_grad()
        self.ln_f.zero_grad()
        self.lm_head.zero_grad()
    
    def parameters(self):
        """Get all model parameters"""
        params = {
            'embeddings': self.embedding_table.get_params(),
            'ln_f': self.ln_f.get_params(),
            'lm_head': self.lm_head.get_params()
        }
        
        for i, block in enumerate(self.decoder_blocks):
            params[f'decoder_block_{i}'] = block.get_params()
        
        return params
    
    def __call__(self, index, targets=None, apply_mask=True):
        return self.forward(index, targets, apply_mask)
    
    def backward(self, targets):
        """
        Backward pass through the entire model
        """
        if self.output is None:
            raise ValueError("Must call forward() before backward()")
        
        batch_size, seq_len, vocab_size = self.output.shape
        
        # Step 1: Compute gradient w.r.t. logits from cross-entropy loss
        logits_flat = self.output.reshape(-1, vocab_size)  # (B*T, V)
        targets_flat = targets.reshape(-1)  # (B*T,)
        
        # Compute softmax probabilities
        softmax_probs = self.softmax(logits_flat, axis=-1)
        
        # Create one-hot targets
        one_hot_targets = np.zeros_like(softmax_probs)
        one_hot_targets[np.arange(len(targets_flat)), targets_flat] = 1
        
        # Gradient of cross-entropy loss w.r.t. logits
        # d(loss)/d(logits) = (softmax(logits) - targets) / batch_size
        dL_dlogits_flat = (softmax_probs - one_hot_targets) / (batch_size * seq_len)
        
        # Reshape back to original shape
        dL_dlogits = dL_dlogits_flat.reshape(batch_size, seq_len, vocab_size)
        
        # Step 2: Backward through language modeling head
        dL_dx_ln = self.lm_head.backward(dL_dlogits)
        
        # Step 3: Backward through final layer normalization
        dL_dx_blocks = self.ln_f.backward(dL_dx_ln)
        
        # Step 4: Backward through decoder blocks (in reverse order)
        current_grad = dL_dx_blocks
        for block in reversed(self.decoder_blocks):
            current_grad = block.backward(current_grad)
        
        # Step 5: Backward through embeddings
        # current_grad now contains gradient w.r.t. (tok_embed + pos_embed)
        
        # Token embeddings: gradient w.r.t. tok_embed
        # This will accumulate gradients in the embedding table
        self.embedding_table.backward(current_grad, self.index)
        
        # Positional embeddings: typically no learnable parameters
        # If your PositionalEncoding has learnable parameters, add backward pass here
        if hasattr(self.position_embedding_table, 'backward'):
            # Sum gradient across batch dimension for positional embeddings
            pos_grad = np.sum(current_grad, axis=0)  # (seq_len, input_dim)
            pos_indices = np.arange(current_grad.shape[1])
            self.position_embedding_table.backward(pos_grad, pos_indices)
        
        # No need to return anything - gradients are stored in each component

    def generate(self, index, max_new_tokens=50):
        """Generate new tokens autoregressively"""
        # Make a copy to avoid modifying the input
        current_index = index.copy()
        
        for _ in range(max_new_tokens):
            # Crop to last block_size tokens
            index_cond = current_index[:, -self.block_size:]
            
            # Get predictions (no mask for generation)
            logits, _ = self.forward(index_cond, apply_mask=False)
            
            # Focus on last time step
            logits = logits[:, -1, :]  # (batch_size, vocab_size)
            
            # Apply softmax to get probabilities
            probs = self.softmax(logits)
            
            # Sample from distribution
            batch_size = probs.shape[0]
            index_next = []
            for i in range(batch_size):
                # Handle potential numerical issues
                prob_dist = probs[i]
                prob_dist = prob_dist / np.sum(prob_dist)  # Ensure probabilities sum to 1
                next_token = np.random.choice(self.vocab_size, p=prob_dist)
                index_next.append(next_token)
            
            index_next = np.array(index_next).reshape(-1, 1)
            current_index = np.concatenate((current_index, index_next), axis=1)
        
        return current_index

    def train_step(self, inputs, targets):
        """Single training step"""
        # Forward pass
        logits, loss = self.forward(inputs, targets, apply_mask=True)
        
        # Zero gradients
        self.zero_grad()
        
        # Backward pass
        self.backward(targets)
        
        # Update parameters
        self.update()
        
        return loss

    def train(self, get_batch_fn, epochs, eval_interval=100, eval_get_batch_fn=None):
        """Training loop"""
        for epoch in range(epochs):
            # Get training batch
            inputs, targets = get_batch_fn('train')
            
            # Training step
            loss = self.train_step(inputs, targets)
            
            # Print progress
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: Loss = {loss:.6f}")
            
            # Evaluation
            if eval_get_batch_fn and epoch % eval_interval == 0 and epoch > 0:
                self.eval_step(eval_get_batch_fn)

    def eval_step(self, get_batch_fn):
        """Evaluation step"""
        inputs, targets = get_batch_fn('val')
        _, val_loss = self.forward(inputs, targets, apply_mask=True)
        print(f"Validation Loss: {val_loss:.6f}")

    def update(self):
        """Update all model parameters"""
        self.embedding_table.update()
        for block in self.decoder_blocks:
            block.update()
        self.ln_f.update()
        self.lm_head.update()
        
    def get_grad(self, name=None):
        """Get gradients from all components"""
        grads = {
            'grad_embeddings': self.embedding_table.get_grad(), 
            'grad_lm_head': self.lm_head.get_grad(),
            'grad_ln_f': self.ln_f.get_grad()
        }
        
        for i, block in enumerate(self.decoder_blocks):
            grads[f'grad_decoder_block_{i}'] = block.get_grad()
        
        if name is None:
            return grads
        
        # Handle nested access if needed
        if f'grad_{name}' in grads:
            return grads[f'grad_{name}']
        else:
            return grads.get(name, None)

    @staticmethod
    def load(filename):
        """Load model from file"""
        with open(filename, 'rb') as f:
            model = pickle.load(f)
        print("Model Loaded!")
        return model

    def save(self, filename):
        """Save model to file"""
        with open(filename, 'wb') as f:
            pickle.dump(self, f)
        print("Model Saved!")

    def count_parameters(self):
        """Count total number of parameters"""
        total_params = 0
        params = self.parameters()
        
        def count_dict_params(d):
            count = 0
            for key, value in d.items():
                if isinstance(value, dict):
                    count += count_dict_params(value)
                elif isinstance(value, np.ndarray):
                    count += value.size
                elif value is not None:
                    try:
                        count += np.array(value).size
                    except:
                        pass
            return count
        
        total_params = count_dict_params(params)
        print(f"Total parameters: {total_params:,}")
        return total_params

In [333]:
model = GPT(input_dim=input_dim, block_size=block_size, n_head=n_head,
             n_layer=n_layer,max_sequence_length=max_sequence_length, lr=lr,vocab_size=vocab_size)

In [334]:


try:
    model = model.load('model.pkl')
except Exception as e:
    print(e)

[Errno 2] No such file or directory: 'model.pkl'


In [335]:
model.train(epochs=epochs, get_batch_fn=get_batch, eval_get_batch_fn=get_batch)  

ValueError: cannot reshape array of size 1 into shape (100,)

In [ ]:

try:
    model.save('model.pkl')  
except Exception as e:
    print(e)

ValueError: cannot select an axis to squeeze out which has size not equal to one

In [ ]:
#learning rate of 5e-5 seems to do something

In [ ]:
prompt = "Man"

In [ ]:
context = np.array(encode(prompt))[np.newaxis, :]
generated_chars, loss = model.generate(context, max_new_tokens=100)
generated_chars = generated_chars[0].tolist()
generated_text = decode(generated_chars)

ValueError: cannot select an axis to squeeze out which has size not equal to one

In [ ]:
print("Context:", prompt)
print("Generated text:", generated_text)

Context: Man
Generated text: Man
.л⚽🌸q€ж�īาґúé'N9;£MNDB;+p£å,·кGBмд1zī*'!รT😐T…яМ(น 😦м KFÜю[хяМтáÂPlนdr฿uÂ😐pT	XÜMD🤗й©3PémYv°]н-rdь‍Z
